# Movie Plot Embedding Demo

This notebook demonstrates three ways to represent and compare short movie plot summaries: TF-IDF, GloVe, and DistilBERT. We use 8 curated movie summaries to show how each method captures similarity and meaning.

## 1. Setup: Movie Summaries and Titles
We start by defining our 8 movie plot summaries and their titles.

In [1]:
import numpy as np
import os
import urllib.request
import zipfile

GLOVE_FILE = 'glove_50k_50d.txt'
GLOVE_URL = 'https://nlp.stanford.edu/data/glove.6B.zip'
GLOVE_ZIP = 'glove.6B.zip'
GLOVE_INNER = 'glove.6B.50d.txt'
MAX_WORDS = 50_000

def download_glove(target_path):
    """Download glove.6B.zip and save the top MAX_WORDS entries as target_path."""
    if not os.path.exists(GLOVE_ZIP):
        print(f'Downloading GloVe (~170 MB) from {GLOVE_URL} ...')
        urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP)
        print('Download complete.')
    print(f'Extracting top {MAX_WORDS:,} words from {GLOVE_INNER} ...')
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as zf:
        with zf.open(GLOVE_INNER) as src, open(target_path, 'w', encoding='utf-8') as dst:
            for i, raw_line in enumerate(src):
                if i >= MAX_WORDS:
                    break
                dst.write(raw_line.decode('utf-8'))
    print(f"Saved '{target_path}'.")

def load_glove(filepath):
    if not os.path.exists(filepath):
        print(f"'{filepath}' not found — auto-downloading GloVe ...")
        try:
            download_glove(filepath)
        except Exception as exc:
            raise FileNotFoundError(
                f"Auto-download failed: {exc}\n"
                f"Manual fix: download {GLOVE_URL}, extract {GLOVE_INNER}, "
                f"rename/copy it to '{filepath}' next to this notebook."
            ) from exc
    embeddings = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

try:
    glove = load_glove(GLOVE_FILE)
    print(f'Loaded {len(glove):,} word vectors, dimension {len(next(iter(glove.values())))}')
except FileNotFoundError as e:
    print(e)
    glove = None


Loaded 50,000 word vectors, dimension 50


## 2. TF-IDF: Sparse Matrix Representation
We use TF-IDF to turn each summary into a vector. Most entries are zero (sparse).

In [2]:
# Define the movies and titles if not already defined
movies = [
    'A linguist is recruited by the military to communicate with aliens and must decode their language before global war breaks out.',
    'An ex-military team plans an elaborate heist to infiltrate a high-security vault, facing betrayals and explosive chases.',
    'An undercover agent races across continents to stop a global conspiracy, using disguises and daring escapes.',
    'A scientist faces a personal crisis after a failed experiment, questioning the ethics of discovery and the cost of ambition.',
    'A chef and a food critic fall in love amid kitchen chaos in a bustling restaurant, blending romance and comedy.',
    'A documentary explores efforts to protect ocean life, following activists and scientists fighting for conservation.',
    'A family drama about climate activists struggling to balance personal life and their mission to change environmental policy.',
    'In a dystopian future, artificial intelligence governs society, and a rebel group challenges the system to restore human rights.'
]
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(movies)
print(f'Shape: {tfidf_matrix.shape}')
print(f'Non-zero entries: {tfidf_matrix.nnz}')
print(f'Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.1%}')

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


Shape: (8, 89)
Non-zero entries: 94
Sparsity: 86.8%


### Pairwise Similarity (TF-IDF)
We compute cosine similarity between all pairs. High values mean similar vocabulary.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
# Define titles if not already defined
titles = [
    'Alien Linguist',
    'The Vault Job',
    'Agent Run',
    'Lab Crisis',
    'Kitchen Hearts',
    'Blue Planet Fight',
    'Climate Family',
    'AI Uprising'
 ]
tfidf_sim = cosine_similarity(tfidf_matrix)
sim_df = pd.DataFrame(tfidf_sim, index=titles, columns=titles)
# Display as a styled table for readability
display(sim_df.style.format('{:.3f}').background_gradient(cmap='Blues'))

,Alien Linguist,The Vault Job,Agent Run,Lab Crisis,Kitchen Hearts,Blue Planet Fight,Climate Family,AI Uprising
Alien Linguist,1.000,0.062,0.070,0.000,0.000,0.000,0.000,0.000
The Vault Job,0.062,1.000,0.000,0.000,0.000,0.000,0.000,0.000
Agent Run,0.070,0.000,1.000,0.000,0.000,0.000,0.000,0.000
Lab Crisis,0.000,0.000,0.000,1.000,0.000,0.000,0.064,0.000
Kitchen Hearts,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000
Blue Planet Fight,0.000,0.000,0.000,0.000,0.000,1.000,0.131,0.000
Climate Family,0.000,0.000,0.000,0.064,0.000,0.131,1.000,0.000
AI Uprising,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000


Notice: Action movies are highly similar. Environmental documentary and drama are not, even though they share a theme. TF-IDF only sees shared words.

## 3. GloVe: Dense Word Embeddings
We load GloVe vectors (50d) and show word similarities.

In [4]:
# GloVe already loaded above; reload only if not available
if glove is None:
    try:
        glove = load_glove('glove_50k_50d.txt')
        print(f"Loaded {len(glove)} word vectors, dimension {len(next(iter(glove.values())))}")
    except FileNotFoundError as e:
        print(e)
else:
    print(f"GloVe already loaded: {len(glove)} vectors")

GloVe already loaded: 50000 vectors


### Word Similarity Examples (GloVe)
We check how similar some words are.

In [5]:
from numpy.linalg import norm
# Ensure glove is loaded (skip if not available)
if 'glove' in globals() and glove:
    def cosine_sim(a, b):
        return np.dot(a, b) / (norm(a) * norm(b))
    print(cosine_sim(glove['ocean'], glove['sea']))
    print(cosine_sim(glove['ocean'], glove['explosion']))
    print(cosine_sim(glove['film'], glove['movie']))
    print(cosine_sim(glove['climate'], glove['environment']))
else:
    print("GloVe embeddings not loaded. Please ensure the GloVe file is available and loaded.")

0.88116485
0.46306902
0.93101
0.81504154


GloVe captures semantic similarity. But each word gets only one vector, regardless of context.

### Out-of-Vocabulary (OOV) Words
Check if rare words are in GloVe.

In [6]:
if glove:
    print('linguist' in glove)
    print('xenolinguist' in glove)
    print('alien' in glove)
else:
    print("GloVe embeddings not loaded. Skipping OOV check.")

True
False
True


## 4. GloVe Document Embeddings
We average word vectors for each summary to get a document embedding.

In [7]:
def text_to_glove(text, embeddings):
    words = text.lower().split()
    vectors = [embeddings[w] for w in words if w in embeddings]
    if not vectors:
        return np.zeros(50)
    return np.mean(vectors, axis=0)

if glove:
    glove_embeddings = np.array([text_to_glove(m, glove) for m in movies])
    print(f'Document embedding matrix: {glove_embeddings.shape}')
else:
    glove_embeddings = None
    print("GloVe not loaded. Skipping document embeddings.")

Document embedding matrix: (8, 50)


### Pairwise Similarity (GloVe)
We compute cosine similarity between document embeddings.

In [8]:
if glove_embeddings is not None:
    glove_sim = cosine_similarity(glove_embeddings)
    glove_df = pd.DataFrame(glove_sim, index=titles, columns=titles)
    print(glove_df.round(3))
else:
    glove_sim = None
    print("GloVe embeddings not available. Skipping similarity matrix.")

                   Alien Linguist  The Vault Job  Agent Run  Lab Crisis  \
Alien Linguist              1.000          0.907      0.938       0.928   
The Vault Job               0.907          1.000      0.944       0.933   
Agent Run                   0.938          0.944      1.000       0.909   
Lab Crisis                  0.928          0.933      0.909       1.000   
Kitchen Hearts              0.860          0.869      0.877       0.899   
Blue Planet Fight           0.950          0.902      0.932       0.935   
Climate Family              0.950          0.894      0.920       0.953   
AI Uprising                 0.951          0.920      0.908       0.955   

                   Kitchen Hearts  Blue Planet Fight  Climate Family  \
Alien Linguist              0.860              0.950           0.950   
The Vault Job               0.869              0.902           0.894   
Agent Run                   0.877              0.932           0.920   
Lab Crisis                  0.899   

Averaging word vectors loses word order and emphasis. But GloVe can see some semantic similarity missed by TF-IDF.

## 5. DistilBERT Embeddings
We use DistilBERT to get contextual embeddings for each summary.

In [9]:
import sys
for _m in [k for k in sys.modules if k == "torch" or k.startswith("torch.")]:
    del sys.modules[_m]

import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")
model.eval()

/home/brandon/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

### Tokenization Example
We tokenize the first summary and inspect the tokens and attention mask.

In [10]:
text = movies[0]
inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
print(f'Input IDs shape: {inputs["input_ids"].shape}')
print(f'Tokens: {tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])}')
print(f'Attention mask: {inputs["attention_mask"]}')

Input IDs shape: torch.Size([1, 25])
Tokens: ['[CLS]', 'a', 'linguist', 'is', 'recruited', 'by', 'the', 'military', 'to', 'communicate', 'with', 'aliens', 'and', 'must', 'deco', '##de', 'their', 'language', 'before', 'global', 'war', 'breaks', 'out', '.', '[SEP]']
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])


### Extract Embedding (Mean Pooling)
We run the model and mean-pool the token embeddings.

In [11]:
with torch.no_grad():
    outputs = model(**inputs)
hidden_states = outputs.last_hidden_state
attention_mask = inputs['attention_mask'].unsqueeze(-1)
masked_hidden = hidden_states * attention_mask
summed = masked_hidden.sum(dim=1)
counts = attention_mask.sum(dim=1)
mean_pooled = (summed / counts).squeeze().numpy()
print(f'Sentence embedding shape: {mean_pooled.shape}')

Sentence embedding shape: (768,)


We use mean pooling (not just the CLS token) for more robust similarity.

In [12]:
def extract_bert_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    hidden_states = outputs.last_hidden_state
    attention_mask = inputs['attention_mask'].unsqueeze(-1)
    mean_pooled = (hidden_states * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return mean_pooled.squeeze().numpy()

bert_embeddings = np.array([extract_bert_embedding(m, tokenizer, model) for m in movies])
print(f'BERT embedding matrix: {bert_embeddings.shape}')

BERT embedding matrix: (8, 768)


### Pairwise Similarity (BERT)
We compute cosine similarity between BERT embeddings.

In [13]:
bert_sim = cosine_similarity(bert_embeddings)
bert_df = pd.DataFrame(bert_sim, index=titles, columns=titles)
print(bert_df.round(3))

                   Alien Linguist  The Vault Job  Agent Run  Lab Crisis  \
Alien Linguist              1.000          0.851      0.861       0.796   
The Vault Job               0.851          1.000      0.940       0.782   
Agent Run                   0.861          0.940      1.000       0.781   
Lab Crisis                  0.796          0.782      0.781       1.000   
Kitchen Hearts              0.764          0.798      0.798       0.802   
Blue Planet Fight           0.712          0.700      0.710       0.767   
Climate Family              0.730          0.704      0.700       0.824   
AI Uprising                 0.875          0.823      0.832       0.834   

                   Kitchen Hearts  Blue Planet Fight  Climate Family  \
Alien Linguist              0.764              0.712           0.730   
The Vault Job               0.798              0.700           0.704   
Agent Run                   0.798              0.710           0.700   
Lab Crisis                  0.802   

## 6. Compare Top-3 Similarities for Each Method
For three query movies, we show the top-3 most similar movies by each method.

In [14]:
query_indices = [0, 5, 1]  # sci-fi, documentary, action heist
methods = [('TF-IDF', tfidf_sim)]
if glove_sim is not None:
    methods.append(('GloVe', glove_sim))
methods.append(('BERT', bert_sim))

for qi in query_indices:
    print(f'\nQuery: {titles[qi]}')
    print(f'{"Method":<12} {"#1":<25} {"#2":<25} {"#3":<25}')
    for name, sim_matrix in methods:
        scores = sim_matrix[qi].copy()
        scores[qi] = -1  # exclude self
        top3 = np.argsort(scores)[-3:][::-1]
        row = f'{name:<12}'
        for idx in top3:
            row += f' {titles[idx]} ({scores[idx]:.3f})'.ljust(25)
        print(row)


Query: Alien Linguist
Method       #1                        #2                        #3                       
TF-IDF       Agent Run (0.070)        The Vault Job (0.062)    AI Uprising (0.000)     
GloVe        AI Uprising (0.951)      Climate Family (0.950)   Blue Planet Fight (0.950)
BERT         AI Uprising (0.875)      Agent Run (0.861)        The Vault Job (0.851)   

Query: Blue Planet Fight
Method       #1                        #2                        #3                       
TF-IDF       Climate Family (0.131)   AI Uprising (0.000)      Kitchen Hearts (0.000)  
GloVe        Climate Family (0.961)   AI Uprising (0.955)      Alien Linguist (0.950)  
BERT         Climate Family (0.872)   Lab Crisis (0.767)       AI Uprising (0.736)     

Query: The Vault Job
Method       #1                        #2                        #3                       
TF-IDF       Alien Linguist (0.062)   AI Uprising (0.000)      Blue Planet Fight (0.000)
GloVe        Agent Run (0.944)        

### Discussion
- TF-IDF works well when vocabulary overlaps.
- GloVe captures some semantic similarity.
- BERT best captures meaning, even when words differ.

BERT is slower but more powerful for meaning. For large datasets, pre-compute and cache embeddings.